# Analisi degli errori di retrieval

Questo notebook analizza i casi in cui la semantic search baseline non recupera la fattura corretta al primo posto.

Input:
- `data/processed/semantic_search_results.jsonl`

Obiettivi:
- separare fallimenti e successi top-1;
- distribuire i fallimenti per `noise_level`, `error_types` e `changed_fields`;
- calcolare un tasso di fallimento per tipo di errore e campo modificato;
- ispezionare manualmente i casi peggiori;
- calcolare metriche separate per `noise_level`.

## 1. Import e percorsi

In [ ]:
import json
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


pd.set_option("display.max_colwidth", 120)

current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
DATA_DIR = PROJECT_ROOT / "data" / "processed"

RESULTS_PATH = DATA_DIR / "semantic_search_results.jsonl"

RESULTS_PATH

## 2. Caricamento risultati

In [ ]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSON non valido in {path} alla riga {line_number}: {exc}") from exc
    return records


results = read_jsonl(RESULTS_PATH)
print(f"Record totali: {len(results):,}")

## 3. Costruzione DataFrame flat

Estraiamo i campi utili dai metadata e creiamo un DataFrame con una riga per query.

In [ ]:
def flatten_result(record: dict[str, Any]) -> dict[str, Any]:
    metadata = record.get("metadata") or {}
    error_types = metadata.get("error_types") or []
    changed_fields = metadata.get("changed_fields") or []

    # error_types e changed_fields possono essere lista o stringa pipe-separated
    if isinstance(error_types, str):
        error_types = [e for e in error_types.split("|") if e]
    if isinstance(changed_fields, str):
        changed_fields = [f for f in changed_fields.split("|") if f]

    return {
        "record_id": record.get("record_id"),
        "source_image": record.get("source_image"),
        "evaluation_split": record.get("evaluation_split"),
        "noise_level": metadata.get("noise_level"),
        "error_types": error_types,
        "changed_fields": changed_fields,
        "correct_rank": record.get("correct_rank"),
        "reciprocal_rank": record.get("reciprocal_rank"),
        "is_match_top_1": record.get("is_match_top_1"),
        "is_match_top_3": record.get("is_match_top_3"),
        "is_match_top_5": record.get("is_match_top_5"),
    }


df = pd.DataFrame([flatten_result(r) for r in results])

print(f"Shape: {df.shape}")
df.head(3)

## 4. Separazione fallimenti e successi top-1

In [ ]:
failures = df[df["is_match_top_1"] == False].copy()
successes = df[df["is_match_top_1"] == True].copy()

print(f"Successi top-1:  {len(successes):,}  ({len(successes)/len(df)*100:.1f}%)")
print(f"Fallimenti top-1: {len(failures):,}  ({len(failures)/len(df)*100:.1f}%)")

## 5. Distribuzione dei fallimenti per `noise_level`

In [ ]:
noise_order = ["low", "medium", "high"]

noise_total = df["noise_level"].value_counts().rename("totale")
noise_failures = failures["noise_level"].value_counts().rename("fallimenti")

noise_stats = pd.concat([noise_total, noise_failures], axis=1).fillna(0).astype(int)
noise_stats["tasso_fallimento_%"] = (noise_stats["fallimenti"] / noise_stats["totale"] * 100).round(1)
noise_stats = noise_stats.reindex([n for n in noise_order if n in noise_stats.index])

noise_stats

## 6. Tasso di fallimento per tipo di errore

Ogni record puo' avere piu' `error_types`. Esplodiamo la colonna e calcoliamo quante volte ogni tipo di errore appare nei fallimenti rispetto al totale.

In [ ]:
def explode_column(frame: pd.DataFrame, col: str) -> pd.Series:
    """Restituisce una Series con tutti i valori esplosi dalla colonna lista."""
    return frame[col].explode().dropna()


error_total = explode_column(df, "error_types").value_counts().rename("totale")
error_failures = explode_column(failures, "error_types").value_counts().rename("fallimenti")

error_stats = pd.concat([error_total, error_failures], axis=1).fillna(0).astype(int)
error_stats["tasso_fallimento_%"] = (error_stats["fallimenti"] / error_stats["totale"] * 100).round(1)
error_stats = error_stats.sort_values("tasso_fallimento_%", ascending=False)

error_stats

## 7. Tasso di fallimento per `changed_fields`

Stesso approccio, ma sui campi modificati. Risponde a: quando viene toccato il campo X, quanto spesso il retrieval fallisce?

In [ ]:
field_total = explode_column(df, "changed_fields").value_counts().rename("totale")
field_failures = explode_column(failures, "changed_fields").value_counts().rename("fallimenti")

field_stats = pd.concat([field_total, field_failures], axis=1).fillna(0).astype(int)
field_stats["tasso_fallimento_%"] = (field_stats["fallimenti"] / field_stats["totale"] * 100).round(1)
field_stats = field_stats.sort_values("tasso_fallimento_%", ascending=False)

# Mostriamo solo i campi con almeno 5 occorrenze nel totale per evitare tassi su campioni troppo piccoli
field_stats[field_stats["totale"] >= 5].head(30)

## 8. I campi in `embedding_text` vs campi solo in `metadata`

Separiamo i `changed_fields` in due gruppi:
- campi che rientrano nell'`embedding_text` (seller/buyer company_name, address, country, products.description);
- campi che sono solo nei metadata (invoice, payment, tax, seller.vat_no, ecc.).

Questo aiuta a capire se i fallimenti sono causati da errori su contenuto semantico o su campi strutturati che l'embedding non vede.

In [ ]:
EMBEDDING_TEXT_FIELDS = {
    "seller.company_name",
    "buyer.company_name",
    "seller.address",
    "buyer.address",
    "buyer.country",
}


def is_product_description(field: str) -> bool:
    parts = field.split(".")
    return len(parts) == 3 and parts[0] == "products" and parts[2] == "description"


def field_in_embedding_text(field: str) -> bool:
    return field in EMBEDDING_TEXT_FIELDS or is_product_description(field)


def count_embedding_vs_metadata_changes(row: pd.Series) -> dict[str, int]:
    fields = row["changed_fields"] or []
    in_embedding = sum(1 for f in fields if field_in_embedding_text(f))
    only_metadata = sum(1 for f in fields if not field_in_embedding_text(f))
    return {"changed_in_embedding_text": in_embedding, "changed_only_metadata": only_metadata}


change_counts = df.apply(count_embedding_vs_metadata_changes, axis=1, result_type="expand")
df_aug = pd.concat([df, change_counts], axis=1)

# Confronto tra fallimenti e successi
comparison = pd.DataFrame({
    "fallimenti": df_aug[df_aug["is_match_top_1"] == False][["changed_in_embedding_text", "changed_only_metadata"]].mean(),
    "successi": df_aug[df_aug["is_match_top_1"] == True][["changed_in_embedding_text", "changed_only_metadata"]].mean(),
}).round(2)

print("Numero medio di campi modificati per gruppo:")
comparison

## 9. Ispezione manuale dei casi peggiori

I 30 fallimenti con `correct_rank` piu' alto, cioe' i casi in cui la fattura corretta e' finita piu' lontana nella classifica.

In [ ]:
worst_failures = (
    failures
    .sort_values("correct_rank", ascending=False)
    .head(30)[
        ["record_id", "source_image", "evaluation_split", "noise_level",
         "error_types", "changed_fields", "correct_rank"]
    ]
    .reset_index(drop=True)
)

worst_failures

## 10. Metriche per `noise_level`

Top-1 accuracy, top-3 accuracy, top-5 accuracy e MRR separati per livello di rumore.

In [ ]:
def compute_metrics(frame: pd.DataFrame) -> dict[str, Any]:
    if frame.empty:
        return {"query_count": 0, "top_1_accuracy": None, "top_3_accuracy": None,
                "top_5_accuracy": None, "mrr": None}
    return {
        "query_count": len(frame),
        "top_1_accuracy": round(float(frame["is_match_top_1"].mean()), 4),
        "top_3_accuracy": round(float(frame["is_match_top_3"].mean()), 4),
        "top_5_accuracy": round(float(frame["is_match_top_5"].mean()), 4),
        "mrr": round(float(frame["reciprocal_rank"].mean()), 4),
    }


metrics_by_noise = pd.DataFrame([
    {"noise_level": level, **compute_metrics(df[df["noise_level"] == level])}
    for level in noise_order
    if level in df["noise_level"].values
])

metrics_by_noise

## 11. Riepilogo finale

In [ ]:
print("=== RIEPILOGO ANALISI ERRORI ===")
print(f"\nQuery totali: {len(df):,}")
print(f"Fallimenti top-1: {len(failures):,} ({len(failures)/len(df)*100:.1f}%)")

print("\n--- Tasso di fallimento per noise_level ---")
print(noise_stats.to_string())

print("\n--- Top 10 error_types per tasso di fallimento ---")
print(error_stats.head(10).to_string())

print("\n--- Top 10 changed_fields per tasso di fallimento (min 5 occorrenze) ---")
print(field_stats[field_stats["totale"] >= 5].head(10).to_string())

print("\n--- Metriche per noise_level ---")
print(metrics_by_noise.to_string(index=False))